# 📖 Notebook 6: RBAC & Security — Locking Down Your Cluster

So far, we've been running everything as a cluster admin — full access to everything.
In a real company, multiple teams share the same cluster. You need to answer:

- **Who** can access **what** resources? (RBAC)
- **Which** pods can talk to **which** other pods? (NetworkPolicies)
- **What** are pods allowed to do on the host? (Pod Security Standards)

This notebook teaches you to build a secure, multi-team Kubernetes environment.

> **Prerequisites: Notebooks 01–03.** The NetworkPolicy exercises send traffic at the
> `api-gateway` and `user-service` Services in `k8s-lab`.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['minikube', 'kubectl']
INSTALL_HINTS = {
    'minikube': 'https://minikube.sigs.k8s.io/docs/start/  (or `brew install minikube`)',
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
# `!kubectl ...` prints but never fails a cell, so every claim below is also
# checked in Python. The NetworkPolicy checks are conditional on whether your
# CNI enforces policy at all -- see the detection cell further down.
import json
import subprocess

NS = "k8s-lab"


def can_i(verb, resource, namespace, identity):
    """`kubectl auth can-i`, as a bool. The single best RBAC debugging command."""
    cmd = ["kubectl", "auth", "can-i", verb, resource, f"--as={identity}"]
    if namespace:
        cmd += ["-n", namespace]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    assert out in ("yes", "no"), f"unexpected answer from kubectl auth can-i: {out!r}"
    return out == "yes"


def probe(target, from_pod="test-pod", ns="team-alpha", timeout=4):
    """Try one HTTP GET from a pod. Returns (reachable, first line of output)."""
    r = subprocess.run(
        ["kubectl", "exec", "-n", ns, from_pod, "--",
         "wget", "-qO-", f"--timeout={timeout}", target],
        capture_output=True, text=True, timeout=90)
    text = (r.stdout or r.stderr).strip().splitlines()
    return r.returncode == 0, (text[0] if text else "(no output)")


def verdict(label, reachable, expected_when_enforced):
    """Print what happened next to what SHOULD happen, given this cluster's CNI."""
    want = expected_when_enforced if ENFORCES else True  # unenforced: everything works
    mark = "✅" if reachable == want else "❌"
    print(f"  {mark} {label}: reachable={reachable}, expected={want}"
          f"{'' if ENFORCES else '  (CNI does not enforce policy)'}")
    assert reachable == want, (
        f"{label}: expected reachable={want} on this cluster, got {reachable}"
    )


print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- Create **namespaces** as isolation boundaries between teams
- Create **ServiceAccounts**, **Roles**, and **RoleBindings** to control API access
- Test permissions with `kubectl auth can-i`
- Apply **NetworkPolicies** to control pod-to-pod traffic
- Enforce **Pod Security Standards** to block dangerous container configurations
- Tell **Role** from **ClusterRole**, and **RoleBinding** from **ClusterRoleBinding** —
  including the useful case of a ClusterRole bound by a RoleBinding
- Check whether your cluster's CNI actually **enforces** NetworkPolicy
- Set **ResourceQuota** and **LimitRange** to prevent resource starvation

## 🛠️ Setup

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

Make sure minikube is running.

In [ ]:
# Verify cluster
!minikube status
!echo '---'
!kubectl get nodes

## ⚠️ Before the NetworkPolicy Section: Does Your CNI Enforce Policy?

This is the single most important thing to know about NetworkPolicy, and it is almost
never said out loud:

> **A NetworkPolicy is enforced by the CNI plugin, not by Kubernetes.** The API server
> stores the object on *any* cluster. `kubectl get networkpolicy` lists it. `kubectl
> describe` prints your rules back at you. And on a CNI that does not implement policy —
> **including minikube's default `bridge`/`kindnet` setup** — absolutely nothing is
> blocked.

There is no warning, no event, no status field. A team can ship a default-deny policy to
production, see it "applied", and be wide open.

The cell below checks which CNI you are running so you know whether the "should fail"
demos below will really fail.

**If your cluster does not enforce policy**, recreate it with Calico (this deletes the
cluster and everything in it, so you would then re-run notebooks 01–03):

```bash
minikube delete
minikube start --cpus=4 --memory=6144 --driver=docker --cni=calico
```

You can also just read on. Every blocked/allowed claim below is labelled with what you
will see in each case.

In [ ]:
import subprocess

pods = subprocess.run(
    ["kubectl", "get", "pods", "-n", "kube-system",
     "-o", "jsonpath={range .items[*]}{.metadata.name}{'\n'}{end}"],
    capture_output=True, text=True,
).stdout

ENFORCING = ("calico", "cilium", "weave", "antrea", "kube-router")
found = sorted({c for c in ENFORCING if c in pods})

if found:
    ENFORCES = True
    print(f"✅ Policy-capable CNI detected: {', '.join(found)}")
    print("   The 'should be blocked' demos below WILL block.")
else:
    ENFORCES = False
    print("⚠️  No policy-enforcing CNI found in kube-system.")
    print("   NetworkPolicy objects will be CREATED but NOT ENFORCED.")
    print("   Every 'should be blocked' demo below will succeed instead.")
    print()
    print("   That is not a bug in the lab -- it is the single most dangerous")
    print("   NetworkPolicy gotcha, and you are now seeing it first-hand.")
    print()
    print("   To get enforcement:")
    print("     minikube delete")
    print("     minikube start --cpus=4 --memory=6144 --driver=docker --cni=calico")

## 🏢 Step 1: Namespaces as Team Boundaries

A **namespace** is like a virtual cluster inside your cluster. Each team gets their own namespace,
and you can set different permissions, quotas, and policies per namespace.

```
┌─────────────────── Kubernetes Cluster ───────────────────┐
│                                                           │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐      │
│  │  team-alpha  │  │  team-beta   │  │  k8s-lab    │      │
│  │             │  │             │  │             │      │
│  │  pods       │  │  pods       │  │  pods       │      │
│  │  services   │  │  services   │  │  services   │      │
│  │  secrets    │  │  secrets    │  │  secrets    │      │
│  │             │  │             │  │             │      │
│  │  RBAC ✓     │  │  RBAC ✓     │  │  RBAC ✓     │      │
│  │  Quota ✓    │  │  Quota ✓    │  │             │      │
│  │  NetPol ✓   │  │  NetPol ✓   │  │             │      │
│  └─────────────┘  └─────────────┘  └─────────────┘      │
│                                                           │
└───────────────────────────────────────────────────────────┘
```

Let's create two team namespaces:

In [ ]:
# Create team namespaces
!kubectl create namespace team-alpha --dry-run=client -o yaml | kubectl apply -f -
!kubectl create namespace team-beta --dry-run=client -o yaml | kubectl apply -f -

print()
!kubectl get namespaces | grep -E 'NAME|team-|k8s-lab'

### Role vs ClusterRole, RoleBinding vs ClusterRoleBinding

Four objects, two independent questions. Getting this straight removes most RBAC
confusion:

- **Role / ClusterRole** answer *what* is permitted (which apiGroups, resources, verbs).
- **RoleBinding / ClusterRoleBinding** answer *who gets it and where*.

|  | roleRef: Role | roleRef: ClusterRole |
|---|---|---|
| **RoleBinding** (namespaced) | Permissions in that **one namespace**. The everyday case. | ⭐ The rules apply, **scoped down to the RoleBinding's namespace**. |
| **ClusterRoleBinding** (cluster-wide) | ❌ **Invalid** — the API server rejects a ClusterRoleBinding whose roleRef is a Role. | Permissions in **every** namespace, plus cluster-scoped objects. |

The starred cell is the one worth memorising. It lets you define a permission set **once**
as a ClusterRole and hand it out per-namespace. That is exactly how the built-in
`view`, `edit` and `admin` ClusterRoles are meant to be used:

```bash
# Give a team edit rights in THEIR namespace only, reusing the built-in ClusterRole.
kubectl create rolebinding team-alpha-edit \
    --clusterrole=edit --group=team-alpha -n team-alpha
```

That grants nothing outside `team-alpha`. Swap `rolebinding` for `clusterrolebinding` and
you have just given that group edit rights over the entire cluster — a one-word typo with
a very large blast radius.

Two more rules that follow from the model:

- **Cluster-scoped resources need a ClusterRole.** Nodes, PersistentVolumes, Namespaces
  and CustomResourceDefinitions do not live in a namespace, so a Role can never grant
  access to them. If a ClusterRole granting `nodes` is bound with a RoleBinding, the
  `nodes` rule is simply inert.
- **RBAC is purely additive. There is no `deny`.** Permissions are the union of every
  binding that applies to you. You cannot subtract a permission granted elsewhere — you
  can only remove the binding that granted it.

### The Golden Rule
**Least privilege**: give the minimum permissions needed. No wildcards. No `cluster-admin`.

### Create a ServiceAccount

A ServiceAccount is like a user account for a pod. By default, Kubernetes auto-mounts a
token into every pod — we'll disable that for security.

In [ ]:
%%writefile ./rbac-sa.yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: alpha-app
  namespace: team-alpha
automountServiceAccountToken: false

In [ ]:
!kubectl apply -f ./rbac-sa.yaml
print("\n✅ ServiceAccount created with automountServiceAccountToken: false")

### Create a Role: Read-Only Access

This Role allows listing and getting pods and services — but NOT deleting, creating, or modifying them.

In [ ]:
%%writefile ./rbac-role.yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: pod-reader
  namespace: team-alpha
rules:
  - apiGroups: [""]
    resources: ["pods", "services"]
    verbs: ["get", "list", "watch"]
  - apiGroups: ["apps"]
    resources: ["deployments"]
    verbs: ["get", "list", "watch"]

In [ ]:
!kubectl apply -f ./rbac-role.yaml
print("\n✅ Role 'pod-reader' created in team-alpha")

### Create a RoleBinding: Connect the Dots

Now we bind the `alpha-app` ServiceAccount to the `pod-reader` Role.

In [ ]:
%%writefile ./rbac-binding.yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: alpha-app-reader
  namespace: team-alpha
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: pod-reader
subjects:
  - kind: ServiceAccount
    name: alpha-app
    namespace: team-alpha

In [ ]:
!kubectl apply -f ./rbac-binding.yaml
print("\n✅ RoleBinding created: alpha-app → pod-reader")

### Test Permissions with `kubectl auth can-i`

This is the most useful RBAC debugging command. It answers: "Can this identity do this action?"

In [ ]:
SA = "system:serviceaccount:team-alpha:alpha-app"

# ✅ Should be ALLOWED — we gave 'list pods' permission
!echo "Can alpha-app list pods in team-alpha?"
!kubectl auth can-i list pods -n team-alpha --as=$SA

print()

# ❌ Should be DENIED — we did NOT give 'delete pods' permission
!echo "Can alpha-app delete pods in team-alpha?"
!kubectl auth can-i delete pods -n team-alpha --as=$SA

print()

# ❌ Should be DENIED — no access to other namespaces
!echo "Can alpha-app list pods in team-beta?"
!kubectl auth can-i list pods -n team-beta --as=$SA

print()

# ❌ Should be DENIED — no access to secrets
!echo "Can alpha-app list secrets in team-alpha?"
!kubectl auth can-i list secrets -n team-alpha --as=$SA

# Least privilege is a claim about four answers, so check all four. An RBAC
# mistake that hands out more than intended is silent by construction -- nothing
# in the cluster reports it, which is why this check belongs in your CI.
expected = {
    ("list", "pods", "team-alpha"): True,     # granted by the Role
    ("delete", "pods", "team-alpha"): False,  # read-only Role
    ("list", "pods", "team-beta"): False,     # a Role is namespace-scoped
    ("list", "secrets", "team-alpha"): False, # secrets are not in the Role
}
for (verb, resource, ns), want in expected.items():
    got = can_i(verb, resource, ns, SA)
    assert got == want, f"can-i {verb} {resource} -n {ns}: expected {want}, got {got}"
print("\n✅ all four answers match least privilege")

**Expected results:**
- `list pods` in team-alpha → **yes** ✅
- `delete pods` in team-alpha → **no** ❌ (read-only role)
- `list pods` in team-beta → **no** ❌ (Role is namespace-scoped)
- `list secrets` in team-alpha → **no** ❌ (secrets not in our Role)

This is **least privilege** in action: the ServiceAccount can only read pods and services in its own namespace.

`kubectl auth can-i --list` is the other half of this tool: instead of asking about one
verb, it dumps every permission an identity has in a namespace. It is the fastest way to
audit "what can this ServiceAccount actually do?"

In [ ]:
# The full picture, rather than one question at a time.
!echo "Everything alpha-app can do in team-alpha:"
!kubectl auth can-i --list -n team-alpha --as=system:serviceaccount:team-alpha:alpha-app

### The ClusterRole-bound-by-a-RoleBinding Case

Let's prove the starred cell from the table above. `view` is a **ClusterRole** that ships
with every cluster. We bind it with a **RoleBinding** in `team-beta` and then check what
that actually granted.

In [ ]:
%%writefile ./clusterrole-scoped-binding.yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: beta-viewer
  namespace: team-beta
---
# roleRef points at the built-in CLUSTER role `view`...
# ...but this is a RoleBinding, so the grant stops at the team-beta boundary.
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: beta-viewer-view
  namespace: team-beta
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: view
subjects:
  - kind: ServiceAccount
    name: beta-viewer
    namespace: team-beta

In [ ]:
!kubectl apply -f ./clusterrole-scoped-binding.yaml

SA = "system:serviceaccount:team-beta:beta-viewer"

print("\n1. list pods in team-beta   (its own namespace)      -> expect yes")
!kubectl auth can-i list pods -n team-beta --as=$SA

print("\n2. list pods in team-alpha  (a different namespace)  -> expect no")
!kubectl auth can-i list pods -n team-alpha --as=$SA

print("\n3. list nodes  (cluster-scoped; `view` does not grant it anyway) -> expect no")
!kubectl auth can-i list nodes --as=$SA

print("\n4. list secrets in team-beta (`view` deliberately excludes Secrets) -> expect no")
!kubectl auth can-i list secrets -n team-beta --as=$SA

# The starred cell of the table, asserted: a ClusterRole bound by a RoleBinding
# grants its rules in exactly one namespace and nowhere else.
assert can_i("list", "pods", "team-beta", SA), \
    "the ClusterRole `view` should apply inside the RoleBinding's own namespace"
assert not can_i("list", "pods", "team-alpha", SA), \
    "a RoleBinding must NOT leak its ClusterRole into other namespaces"
assert not can_i("list", "nodes", None, SA), \
    "cluster-scoped resources cannot be granted by a RoleBinding"
assert not can_i("list", "secrets", "team-beta", SA), \
    "the built-in `view` ClusterRole deliberately excludes Secrets"
print("\n✅ one ClusterRole definition, granted in exactly one namespace")

One cluster-wide role definition, granted in exactly one namespace. Change `kind:
RoleBinding` to `kind: ClusterRoleBinding` (and drop the `namespace`) and answer #2 flips
to **yes** for every namespace in the cluster, including `kube-system`.

Note answer #4: the built-in `view` ClusterRole intentionally omits `secrets`, precisely
because read access to a Secret is read access to its contents. `edit` omits them too.

## 🔒 Step 3: NetworkPolicies — Controlling Pod Traffic

By default, **every pod can talk to every other pod** in the cluster — even across namespaces.
That's dangerous. If an attacker compromises one pod, they can reach everything.

**NetworkPolicies** are firewall rules for pods:

```
Without NetworkPolicy:           With NetworkPolicy:

┌──────┐    ┌──────┐            ┌──────┐    ┌──────┐
│ Pod A │───▶│ Pod B │            │ Pod A │───▶│ Pod B │  ✅ allowed
└──────┘    └──────┘            └──────┘    └──────┘
    │                                │
    ▼                                ▼
┌──────┐                        ┌──────┐
│ Pod C │  ✅ anyone can talk    │ Pod C │  ❌ blocked!
└──────┘                        └──────┘
```

### Best Practice: Default Deny + Explicit Allow

1. Start by denying ALL traffic
2. Then explicitly allow only the traffic your app needs

In [ ]:
# First, deploy a test pod in team-alpha so we can test connectivity.
# `kubectl run` fails with AlreadyExists on a second run; delete-then-create keeps
# the notebook re-runnable from the top.
!kubectl delete pod test-pod -n team-alpha --ignore-not-found --wait=true
!kubectl run test-pod -n team-alpha --image=busybox:1.36 \
    --restart=Never -- sleep 3600

!kubectl wait --for=condition=ready pod/test-pod -n team-alpha --timeout=120s

pod = json.loads(subprocess.run(
    ["kubectl", "get", "pod", "test-pod", "-n", "team-alpha", "-o", "json"],
    capture_output=True, text=True).stdout)
assert pod["status"]["phase"] == "Running", \
    f"the connectivity test pod is {pod['status']['phase']}, not Running"
print("\n✅ Test pod running in team-alpha")

In [ ]:
# Before NetworkPolicy: can test-pod reach pods in k8s-lab?
!echo "Testing connectivity to api-gateway in k8s-lab namespace..."

reachable, out = probe("http://api-gateway.k8s-lab:8000/health")
print(f"  reachable={reachable}  {out}")
assert reachable, (
    "with no NetworkPolicy in place, cross-namespace traffic must work -- if it "
    "does not, something else is broken before the security lesson even starts"
)
print("\n☝️ Without NetworkPolicy, cross-namespace traffic is ALLOWED (by default,"
      "\n   every pod in a cluster can reach every other pod, in any namespace)")

### Apply Default-Deny Policy

This policy blocks ALL ingress and egress traffic for every pod in the namespace.

`podSelector: {}` is an empty selector, which means **every pod**, not *no* pod. Listing a
`policyType` with no matching rule underneath means "allow nothing in that direction".

Two mechanics to hold on to:

1. **There is no deny rule in NetworkPolicy.** You cannot write "block X". You can only
   write allows. A pod selected by *at least one* policy for a direction may only send /
   receive what the **union** of those policies permits. A pod selected by *no* policy is
   completely unrestricted in that direction. "Default deny" is therefore just an empty
   allow-list applied to everything.
2. **Policies are namespaced and select pods, not Services.** They act on pod IPs, so a
   rule allowing `app: api-gateway` allows traffic to those pods however it was addressed
   — through the Service VIP, or directly.

In [ ]:
%%writefile ./default-deny.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-all
  namespace: team-alpha
spec:
  podSelector: {}
  policyTypes:
    - Ingress
    - Egress

In [ ]:
!kubectl apply -f ./default-deny.yaml

import time
time.sleep(3)

print("\n🔒 Default-deny applied to team-alpha")
print("Now the same request again. What SHOULD happen depends on your CNI:")
print()

http_ok, http_out = probe("http://api-gateway.k8s-lab:8000/health")
verdict("HTTP to api-gateway after default-deny", http_ok, expected_when_enforced=False)
print(f"     app said: {http_out}")

print("\n--- and separately: is DNS itself still working? ---")
# Ask for the fully-qualified name: busybox's nslookup does NOT apply the
# `search` domains from /etc/resolv.conf, so a short name would return NXDOMAIN
# even on a completely unrestricted cluster and look like a policy problem.
dns = subprocess.run(
    ["kubectl", "exec", "-n", "team-alpha", "test-pod", "--",
     "nslookup", "api-gateway.k8s-lab.svc.cluster.local"],
    capture_output=True, text=True, timeout=90)
dns_ok = dns.returncode == 0
print(dns.stdout.strip()[-300:] or dns.stderr.strip()[-300:])
verdict("DNS resolution after default-deny", dns_ok, expected_when_enforced=False)

if not ENFORCES:
    print("\n⚠️  Nothing was blocked, because this cluster's CNI does not implement")
    print("    NetworkPolicy. The objects exist, `kubectl get netpol` lists them, and")
    print("    they do nothing. That IS the lesson of this section -- see below.")

### 💥 The Failure Everyone Ships: Default-Deny Kills DNS

Read the two verdicts above before moving on. On a policy-enforcing CNI **both** turn
into failures, and the second one is the interesting one.

Default-deny on **Egress** also blocks UDP/TCP port 53 to CoreDNS. Every pod in the
namespace instantly loses name resolution. And the symptom does not look like a firewall
problem — you get `wget: bad address 'api-gateway.k8s-lab'` or an application-level
`Name or service not known`, so people go hunting for a broken Service or a typo in a
hostname. Meanwhile a policy they added an hour ago is quietly eating every DNS query in
the namespace.

It is so routine that "did you allow egress to port 53?" is the first question anyone
asks about a new default-deny policy. The allow rule you need:

```yaml
egress:
  - ports:
      - port: 53
        protocol: UDP
      - port: 53
        protocol: TCP    # DNS falls back to TCP for large responses
```

Note **both protocols**. Allowing only UDP works right up until a response exceeds 512
bytes and the resolver retries over TCP, which produces an intermittent failure that is
far worse than a consistent one.

An egress rule with `ports` but no `to` means "any destination, these ports only", which
is the pragmatic form. The tighter form targets the CoreDNS pods explicitly:

```yaml
egress:
  - to:
      - namespaceSelector:
          matchLabels:
            kubernetes.io/metadata.name: kube-system
        podSelector:
          matchLabels:
            k8s-app: kube-dns
    ports:
      - port: 53
        protocol: UDP
      - port: 53
        protocol: TCP
```

> **If your CNI does not enforce policy**, both verdicts above said `reachable=True` and
> nothing broke. Do not read that as "my policy is fine". It is the most dangerous
> NetworkPolicy failure mode there is: the policy was accepted, is listed, and protects
> nothing. Notebook 01 shows how to start minikube with `--cni=calico` if you want the
> real thing.

Now let's allow DNS and egress to `k8s-lab`'s api-gateway.

In [ ]:
%%writefile ./allow-dns-and-gateway.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-dns
  namespace: team-alpha
spec:
  podSelector: {}
  policyTypes:
    - Egress
  egress:
    - ports:
        - port: 53
          protocol: UDP
        - port: 53
          protocol: TCP
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-to-gateway
  namespace: team-alpha
spec:
  podSelector: {}
  policyTypes:
    - Egress
  egress:
    - to:
        - namespaceSelector:
            matchLabels:
              kubernetes.io/metadata.name: k8s-lab
          podSelector:
            matchLabels:
              app: api-gateway
      ports:
        - port: 8000

In [ ]:
!kubectl apply -f ./allow-dns-and-gateway.yaml

import time
time.sleep(3)

# The point of the two policies: api-gateway becomes reachable again, and the
# backends behind it do NOT. Both halves are checked -- an "allow" that quietly
# allows everything is not a firewall.
print("Test 1: api-gateway (the one we explicitly allowed)")
gw_ok, gw_out = probe("http://api-gateway.k8s-lab:8000/health")
verdict("api-gateway", gw_ok, expected_when_enforced=True)
print(f"     app said: {gw_out}")

print("\nTest 2: user-service directly (never allowed)")
be_ok, be_out = probe("http://user-service.k8s-lab:8001/health")
verdict("user-service (direct)", be_ok, expected_when_enforced=False)
print(f"     app said: {be_out}")

**Result**: on a policy-enforcing CNI the two verdicts above read
`api-gateway: reachable=True` and `user-service (direct): reachable=False`. The
NetworkPolicy is acting as a firewall: team-alpha pods can reach the front door and
nothing behind it. That is least privilege applied to networking.

On minikube's default CNI both say `reachable=True`, and the assertions above expect
exactly that — the notebook is not pretending otherwise. What the section still teaches
in that case is the more important half: **the objects were created, they are listed, and
they enforce nothing.** There is no status field, no event and no warning to tell you
which situation you are in. The only way to know is to test the traffic, which is what
these cells do.

## 🛡️ Step 4: Pod Security Standards

Kubernetes has three built-in security profiles that prevent pods from doing dangerous things:

| Level | What It Blocks |
|-------|----------------|
| **Privileged** | Nothing — allows everything (development only) |
| **Baseline** | Blocks the most dangerous options: hostNetwork, hostPID, privileged containers |
| **Restricted** | Blocks everything above + requires non-root, read-only filesystem, drops all capabilities |

You apply them using **labels** on namespaces:
- `enforce` → rejects pods that violate the policy
- `warn` → allows but prints a warning
- `audit` → allows but logs in the audit log

In [ ]:
# Apply Pod Security Standards to team-alpha
# enforce=baseline: block dangerous pods
# warn=restricted: show warnings for non-optimal pods
!kubectl label namespace team-alpha \
    pod-security.kubernetes.io/enforce=baseline \
    pod-security.kubernetes.io/warn=restricted \
    --overwrite

print("\n✅ Pod Security Standards applied to team-alpha")
!kubectl get namespace team-alpha --show-labels | grep -o 'pod-security[^ ]*'

In [ ]:
# Try to create a privileged pod — this should be REJECTED
!echo "Attempting to create a privileged pod..."
!kubectl delete pod evil-pod -n team-alpha --ignore-not-found
!kubectl run evil-pod -n team-alpha --image=nginx \
    --overrides='{"spec":{"containers":[{"name":"evil","image":"nginx","securityContext":{"privileged":true}}]}}' \
    2>&1

# "should be REJECTED" is a claim, so check it. A Pod Security label that silently
# does nothing -- misspelled, or on the wrong namespace -- looks exactly like a
# working one until the day it matters.
attempt = subprocess.run(
    ["kubectl", "run", "evil-pod-check", "-n", "team-alpha", "--image=nginx:1.27",
     "--overrides", '{"spec":{"containers":[{"name":"evil","image":"nginx:1.27",'
                    '"securityContext":{"privileged":true}}]}}'],
    capture_output=True, text=True, timeout=120)
assert attempt.returncode != 0, \
    "the privileged pod was ACCEPTED -- Pod Security is not enforcing on team-alpha"
assert "PodSecurity" in attempt.stderr and "privileged" in attempt.stderr, \
    f"rejected, but not by Pod Security: {attempt.stderr[:400]}"
print("\n☝️ The API server refused it:")
print("   " + attempt.stderr.strip().splitlines()[0][:200])
print("\n✅ privileged:true is blocked by enforce=baseline, at admission time --")
print("   the pod never reaches a node, so there is nothing to clean up")

In [ ]:
# A normal pod should still work fine -- `baseline` blocks dangerous options, not
# ordinary containers. (`restricted` would reject this one too, but it is set to
# `warn`, so you get the yellow warning below and the pod is still created.)
!kubectl delete pod good-pod -n team-alpha --ignore-not-found --wait=true
!kubectl run good-pod -n team-alpha --image=nginx:1.27 2>&1
!kubectl wait --for=condition=ready pod/good-pod -n team-alpha --timeout=120s

good = json.loads(subprocess.run(
    ["kubectl", "get", "pod", "good-pod", "-n", "team-alpha", "-o", "json"],
    capture_output=True, text=True).stdout)
assert good["status"]["phase"] == "Running", \
    f"a non-privileged pod should be admitted and run; it is {good['status']['phase']}"
print("\n✅ Normal (non-privileged) pod created and running")

# Clean up
!kubectl delete pod good-pod -n team-alpha --ignore-not-found

## 📊 Step 5: ResourceQuota and LimitRange

Even with RBAC and NetworkPolicies, a team could still consume all cluster resources
and starve other teams. **ResourceQuota** prevents that.

- **ResourceQuota**: hard limits on total CPU, memory, and pod count per namespace
- **LimitRange**: default resource requests/limits for individual pods

In [ ]:
%%writefile ./quota.yaml
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-alpha-quota
  namespace: team-alpha
spec:
  hard:
    requests.cpu: "2"
    requests.memory: 2Gi
    limits.cpu: "4"
    limits.memory: 4Gi
    pods: "20"
---
apiVersion: v1
kind: LimitRange
metadata:
  name: team-alpha-limits
  namespace: team-alpha
spec:
  limits:
    - default:
        cpu: 200m
        memory: 256Mi
      defaultRequest:
        cpu: 100m
        memory: 128Mi
      type: Container

In [ ]:
!kubectl apply -f ./quota.yaml

print("\n✅ ResourceQuota and LimitRange applied to team-alpha")
print("\n--- ResourceQuota ---")
!kubectl describe resourcequota team-alpha-quota -n team-alpha

print("\n--- LimitRange ---")
!kubectl describe limitrange team-alpha-limits -n team-alpha

In [ ]:
# When you create a pod without specifying resources, LimitRange adds defaults:
!kubectl delete pod auto-limits-pod -n team-alpha --ignore-not-found --wait=true
!kubectl run auto-limits-pod -n team-alpha --image=nginx:1.27
!kubectl wait --for=condition=ready pod/auto-limits-pod -n team-alpha --timeout=120s

print("\nNotice the auto-assigned resource requests/limits:")
!kubectl get pod auto-limits-pod -n team-alpha -o jsonpath='{.spec.containers[0].resources}' | python3 -m json.tool

# We asked for no resources at all. The LimitRange mutated the pod on admission,
# which also means it is no longer BestEffort -- and that is the point: a
# LimitRange is how you stop a namespace filling up with unschedulable,
# first-to-be-evicted pods that nobody remembered to size.
pod = json.loads(subprocess.run(
    ["kubectl", "get", "pod", "auto-limits-pod", "-n", "team-alpha", "-o", "json"],
    capture_output=True, text=True).stdout)
res = pod["spec"]["containers"][0]["resources"]
assert res["requests"] == {"cpu": "100m", "memory": "128Mi"}, \
    f"LimitRange defaultRequest was not applied: {res}"
assert res["limits"] == {"cpu": "200m", "memory": "256Mi"}, \
    f"LimitRange default (limit) was not applied: {res}"
assert pod["status"]["qosClass"] == "Burstable", \
    f"defaulted pod should be Burstable, not {pod['status']['qosClass']}"
print("\n✅ a pod created with NO resources came out Burstable with the namespace defaults")

# Clean up
!kubectl delete pod auto-limits-pod -n team-alpha --ignore-not-found

## 🧹 Clean Up

In [ ]:
# Delete the test pod
!kubectl delete pod test-pod -n team-alpha --ignore-not-found
!kubectl delete pod evil-pod evil-pod-check good-pod auto-limits-pod -n team-alpha --ignore-not-found

# Remove NetworkPolicies (so a re-run starts from an unrestricted namespace)
!kubectl delete networkpolicy --all -n team-alpha

# Clean up generated files
!rm -f ./rbac-sa.yaml ./rbac-role.yaml ./rbac-binding.yaml
!rm -f ./default-deny.yaml ./allow-dns-and-gateway.yaml ./quota.yaml
!rm -f ./clusterrole-scoped-binding.yaml

# Keep namespaces, ServiceAccounts, Roles, quotas and LimitRanges: every cell
# above uses `apply`, so the whole notebook is safe to re-run as-is.
print("✅ Cleaned up! (namespaces team-alpha and team-beta kept for later labs)")

## 🎓 What You Learned

In this notebook you:

1. **Created namespaces** as isolation boundaries for different teams
2. **Set up RBAC** with ServiceAccount → Role → RoleBinding to grant read-only access
3. **Tested permissions** with `kubectl auth can-i` — confirmed least-privilege works
4. **Applied NetworkPolicies** — default-deny + explicit allow for specific traffic
5. **Enforced Pod Security Standards** — blocked privileged containers at the namespace level
6. **Set ResourceQuota and LimitRange** — prevented resource starvation between teams

### Key Takeaways

- **Namespaces** are the primary isolation boundary in Kubernetes
- **RBAC** controls API access — always use least privilege, avoid wildcards and `cluster-admin`
- **ClusterRole + RoleBinding** reuses one role definition while scoping the grant to
  a single namespace; **ClusterRoleBinding** grants it everywhere. RBAC is additive —
  there is no deny
- **NetworkPolicies** control network traffic — always start with default-deny, and
  **always allow UDP+TCP egress on port 53** or you have just broken DNS for the
  whole namespace
- A NetworkPolicy is enforced by the **CNI**, not by Kubernetes. On a CNI without
  policy support it is stored, displayed, and completely ignored
- **Pod Security Standards** prevent dangerous container configurations — use `baseline` at minimum
- **ResourceQuota** prevents one team from consuming all cluster resources
- Test everything with `kubectl auth can-i` — it's your best RBAC debugging tool

### Security Checklist for Production Namespaces

| ✅ | Item |
|----|------|
| ☐ | Dedicated namespace per team |
| ☐ | RBAC Role (or ClusterRole) + **RoleBinding** — no ClusterRoleBindings |
| ☐ | ServiceAccount with automountServiceAccountToken: false |
| ☐ | NetworkPolicy default-deny + explicit allows |
| ☐ | Pod Security Standards: baseline (enforce) + restricted (warn) |
| ☐ | ResourceQuota for CPU, memory, pods |
| ☐ | LimitRange for default requests/limits |

### Next Steps

In **Notebook 07** you'll learn GitOps with ArgoCD — deploying and managing your apps
through Git instead of running `kubectl apply` manually.